# Prática: Variational Auto-Encoder (VAE) no CIFAR-10

Neste notebook vamos construir um **VAE** para imagens coloridas do CIFAR-10.

Um VAE tem duas partes:

- **Encoder:** comprime a imagem em uma distribuição no espaço latente (média `z_mean` e log-variância `z_log_var`) e amostra um vetor `z`.
- **Decoder:** reconstrói a imagem a partir de `z`.

A perda total combina **reconstrução** (a imagem de saída deve parecer a de entrada) e **divergência KL** (a distribuição aprendida deve ficar próxima de uma normal padrão), o que permite gerar imagens novas amostrando o espaço latente.

## 1. Importações e semente

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from matplotlib import pyplot as plt

print("GPUs disponíveis: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
SEED = 123456
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Carregando os dados

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

categorias = {
    0: "Avião", 1: "Automóvel", 2: "Pássaro", 3: "Gato", 4: "Cervo",
    5: "Cachorro", 6: "Sapo", 7: "Cavalo", 8: "Navio", 9: "Caminhão"
}

print("Formato do treino: ", x_train.shape)
print("Formato do teste: ", x_test.shape)

In [ ]:
plt.figure(figsize=(9, 9))
rndSamples = np.random.choice(50000, 9)

for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_train[rndSamples[i]])
    plt.title(categorias[int(y_train[rndSamples[i]])])
    plt.axis("off")
plt.show()

In [ ]:
# o VAE é não supervisionado: usamos treino + teste juntos
dataset = np.concatenate([x_train, x_test], axis=0)
dataset = dataset.astype("float32") / 255      # normaliza para [0, 1]

# os rótulos só serão usados para colorir o espaço latente no final
labels = np.concatenate([y_train, y_test], axis=0).flatten()

print(dataset.shape)

## 3. Camada de amostragem (truque da reparametrização)

Para que o gradiente passe pela amostragem aleatória, escrevemos

$$z = \mu + \sigma \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 1)$$

onde $\sigma = \exp(0.5 \cdot \log\sigma^2)$.

In [ ]:
class SamplingLayer(layers.Layer):
    """Truque da reparametrização: z = mu + sigma * epsilon"""

    def call(self, inputs):
        zMean, zLogVar = inputs
        batch = tf.shape(zMean)[0]
        dim = tf.shape(zMean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return zMean + tf.exp(0.5 * zLogVar) * epsilon

## 4. Encoder

In [ ]:
LATENT_DIM = 16

def buildEncoder(latentDim, encoderInputs):
    # bloco convolucional: extrai características e reduz a resolução (32 -> 16 -> 8 -> 4)
    l1 = keras.models.Sequential([
        layers.Conv2D(128, 3, activation="relu", strides=2, padding="same"),
        layers.Conv2D(64, 3, activation="relu", strides=2, padding="same"),
        layers.Conv2D(32, 3, activation="relu", strides=2, padding="same"),
        layers.Flatten(),
        layers.Dense(256, activation="relu")
    ])

    x = l1(encoderInputs)
    zMean = layers.Dense(latentDim, name="z_mean")(x)       # média da distribuição
    zLogVar = layers.Dense(latentDim, name="z_log_var")(x)  # log da variância
    z = SamplingLayer()([zMean, zLogVar])                   # vetor latente amostrado

    return keras.Model(encoderInputs, [zMean, zLogVar, z], name="encoder")

encoderInputs = keras.Input(shape=(32, 32, 3))
encoder = buildEncoder(LATENT_DIM, encoderInputs)
encoder.summary()

## 5. Decoder

Simétrico ao encoder, usando `Conv2DTranspose` para voltar de 4x4 até 32x32. A saída tem 3 canais (RGB) com `sigmoid`.

In [ ]:
def buildDecoder(latentInputs):
    l1 = keras.models.Sequential([
        layers.Dense(4*4*64, activation="relu"),
        layers.Reshape((4, 4, 64)),
        layers.Conv2DTranspose(128, 3, activation="relu", strides=2, padding="same"),  # 8x8
        layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same"),   # 16x16
        layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same"),   # 32x32
        layers.Conv2DTranspose(3, 3, activation="sigmoid", padding="same")             # RGB
    ])

    return keras.Model(latentInputs, l1(latentInputs), name="decoder")

latentInputs = keras.Input(shape=(LATENT_DIM,))
decoder = buildDecoder(latentInputs)
decoder.summary()

## 6. Funções de perda

- **Reconstrução:** entropia cruzada binária entre a imagem original e a reconstruída, somada sobre os pixels.
- **KL:** $-\frac{1}{2}\sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$, mede o quanto a distribuição aprendida se afasta de $\mathcal{N}(0, 1)$.
- **Total:** `reconstrução + klWeight * KL`. O `klWeight` equilibra os dois termos.

In [ ]:
def reconstructionLoss(data, reconstructed):
    # binary_crossentropy faz a média nos canais -> (batch, 32, 32); somamos nos pixels
    return tf.reduce_mean(
        tf.reduce_sum(
            keras.losses.binary_crossentropy(data, reconstructed),
            axis=(1, 2)
        )
    )

def klDivergenceLoss(zMean, zLogVar):
    return tf.reduce_mean(
        tf.reduce_sum(
            -0.5 * (1 + zLogVar - tf.square(zMean) - tf.exp(zLogVar)),
            axis=1
        )
    )

def calcTotalLoss(data, reconstructed, zMean, zLogVar):
    loss1 = reconstructionLoss(data, reconstructed)
    loss2 = klDivergenceLoss(zMean, zLogVar)
    klWeight = 3.0
    return loss1, loss2, loss1 + klWeight * loss2

## 7. Modelo VAE com `train_step` personalizado

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.totalLossTracker = keras.metrics.Mean(name="total_loss")
        self.ceLossTracker = keras.metrics.Mean(name="ce_loss")
        self.klLossTracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.totalLossTracker, self.ceLossTracker, self.klLossTracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            zMean, zLogVar, z = self.encoder(data)
            reconstruction = self.decoder(z)
            ceLoss, klLoss, totalLoss = calcTotalLoss(data, reconstruction, zMean, zLogVar)

        grads = tape.gradient(totalLoss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.totalLossTracker.update_state(totalLoss)
        self.ceLossTracker.update_state(ceLoss)
        self.klLossTracker.update_state(klLoss)

        return {
            "perda_total": self.totalLossTracker.result(),
            "perda_reconstrucao": self.ceLossTracker.result(),
            "perda_kl": self.klLossTracker.result()
        }

## 8. Treinamento

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001))
history = vae.fit(dataset, epochs=32, batch_size=128)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history["perda_total"], label="perda total")
plt.plot(history.history["perda_reconstrucao"], label="perda de reconstrução")
plt.plot(history.history["perda_kl"], label="perda KL")
plt.xlabel("época")
plt.legend()
plt.show()

In [ ]:
# zoom na perda KL
plt.figure(figsize=(10, 6))
plt.plot(history.history["perda_kl"], label="perda KL")
plt.xlabel("época")
plt.legend()
plt.show()

## 9. Reconstrução de uma imagem

Codificamos uma imagem no espaço latente e depois decodificamos para ver o quanto se parece com a original.

In [ ]:
idx = 42
batch = np.expand_dims(x_train[idx], axis=0).astype("float32") / 255
print("Formato do lote: ", batch.shape)
print("Categoria: ", categorias[int(y_train[idx])])

_, _, z = vae.encoder.predict(batch)
synth = vae.decoder.predict(z)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1); plt.imshow(batch[0]); plt.title("Original"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(synth[0]); plt.title("Reconstruída"); plt.axis("off")
plt.show()

print("Vetor latente z:", z)

## 10. Gerando imagens novas

Amostramos pontos aleatórios do espaço latente e passamos pelo decoder.

In [ ]:
z = np.random.normal(loc=0, scale=1, size=(100, LATENT_DIM))
synth = vae.decoder.predict(z)

plt.figure(figsize=(12, 12))
for i in range(100):
    plt.subplot(10, 10, i+1)
    plt.imshow(synth[i])
    plt.axis("off")
plt.show()

## 11. Espaço latente por categoria

Projetamos as médias `z_mean` das duas primeiras dimensões e colorimos pela categoria.

In [ ]:
zMeanAll, _, _ = vae.encoder.predict(dataset)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(zMeanAll[:, 0], zMeanAll[:, 1], c=labels, cmap="tab10", alpha=0.5, s=1)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.ax.set_yticklabels([categorias[i] for i in range(10)])
plt.xlabel("z_mean[0]")
plt.ylabel("z_mean[1]")
plt.title("Espaço latente do VAE — CIFAR-10")
plt.show()

## Exercícios

1. Altere `LATENT_DIM` (4, 32, 64). Como isso afeta a reconstrução e a perda KL?
2. Mude o `klWeight` na função de perda (por exemplo 0.5 e 10). O que acontece com as imagens geradas?
3. Altere o `scale` da amostragem no item 10 (1 → 4). Por que as imagens ficam piores?
4. Interpole entre os vetores latentes de duas imagens diferentes e visualize o resultado.